In [0]:
%sql
GRANT ALL PRIVILEGES ON CATALOG novamart TO `data_engineer_team`;

GRANT USAGE ON CATALOG novamart TO `data_analytics_group`;
GRANT USAGE ON SCHEMA novamart.gold TO `data_analyst_team`;
GRANT SELECT ON SCHEMA novamart.gold TO `data_analyst_team`;

In [0]:
%sql
CREATE OR REPLACE FUNCTION novamart.gold.tag_email_mask_pii(val STRING)
RETURNS STRING
RETURN CASE
    WHEN IS_MEMBER('admin_group') THEN val
    ELSE 'CONFIDENTIAL DATA'
END;

CREATE OR REPLACE POLICY mask_pii_policy
ON CATALOG novamart
COLUMN MASK novamart.gold.tag_email_mask_pii
TO `acoount users`
FOR TABLES MATCH COLUMNS has_tag('sensitivity') AS col ON COLUMN col;

In [0]:
%sql
ALTER TABLE novamart.gold.dim_customer 
ALTER COLUMN email SET TAGS ('sensitivity' = 'PII');

In [0]:
%sql
CREATE OR REPLACE FUNCTION novamart.gold.filter_region(region STRING)
RETURNS BOOLEAN
RETURN CASE
  WHEN IS_MEMBER('admin_group') THEN TRUE
  WHEN IS_MEMBER('latam_managers') AND region = 'LATAM' THEN TRUE
  WHEN IS_MEMBER('apac_managers') AND region = 'APAC' THEN TRUE
  ELSE FALSE
END;

ALTER TABLE novamart.gold.dim_customer SET ROW FILTER novamart.gold.filter_region ON (region);